# Harris County Monthly Runner - Colab Smoke Test

This notebook runs the Harris County monthly scraper through the project CLI. It does not duplicate scraper logic.

Safety notes:

- Start with `--limit 1`.
- Do not run the full month repeatedly.
- Generated data may include public-record information such as names, addresses, document IDs, loan amounts, and legal descriptions.
- Do not commit generated PDFs, text cache files, CSVs, JSONL files, checkpoints, logs, diagnostics, or failed-record files without review.
- Only Harris County is implemented right now. Other counties and app/Streamlit support are future work.

## 1. Clone The GitHub Repo

This clones the public repo into the Colab runtime. If the folder already exists, it fetches the latest refs instead of recloning.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d crapper ]; then
  git clone https://github.com/HeritageRealtySolutions/crapper.git
else
  cd crapper
  git fetch --all --prune
  git status --short
fi

## 2. Install Dependencies

Install Python dependencies from `requirements_free.txt`.

In [ ]:
%%bash
set -e
cd /content/crapper
python -m pip install -r requirements_free.txt

## 3. Install Playwright Chromium

The Harris runner uses Playwright for the search and PDF download flow.

In [ ]:
%%bash
set -e
cd /content/crapper
python -m playwright install chromium

## 4. Run A Limit-1 Harris Smoke Test

This calls the project CLI. Keep `--limit 1` until you have confirmed the workflow is behaving correctly.

In [ ]:
%%bash
set -e
cd /content/crapper
python -m scraper.main --county harris --year 2026 --month 5 --limit 1

## 5. Display The CSV

Load the generated CSV with pandas for inspection.

In [ ]:
from pathlib import Path
import pandas as pd

repo = Path('/content/crapper')
csv_path = repo / 'data/outputs/harris_2026_05_foreclosures.csv'

df = pd.read_csv(csv_path)
display(df)

## 6. Show Row Count And Doc IDs

In [ ]:
print(f'Rows: {len(df)}')
if 'doc_id' in df.columns:
    print('Doc IDs:')
    for doc_id in df['doc_id'].astype(str).tolist():
        print(f'- {doc_id}')
else:
    print('No doc_id column found')

## 7. Show Output Paths

In [ ]:
paths = [
    repo / 'data/outputs/harris_2026_05_foreclosures.csv',
    repo / 'data/outputs/harris_2026_05_foreclosures.jsonl',
    repo / 'data/checkpoints/harris_2026_05_checkpoint.json',
    repo / 'data/failed/harris_2026_05_failed.json',
    repo / 'data/text_cache/harris/2026/05',
    repo / 'data/pdfs/harris/2026/05',
]

for path in paths:
    print(f'{path}: exists={path.exists()}')

## 8. Download The CSV From Colab

This downloads only the CSV. Review generated data before sharing or committing it.

In [ ]:
from google.colab import files

files.download(str(csv_path))